# Baseline Model Experiments

Goals:
- Train Logistic Regression baseline
- Train XGBoost final model
- Compare metrics
- Visualise calibration

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
from src.ingestion.load_data import load_raw
from src.processing.preprocess import clean_raw, build_scaler, apply_scaler
from src.features.feature_store import add_derived_features
from src.processing.split import split_dataset, get_X_y

df_raw = load_raw()
df = add_derived_features(clean_raw(df_raw))
df_train, df_val, df_test = split_dataset(df)

X_train_raw, y_train = get_X_y(df_train)
X_val_raw, y_val   = get_X_y(df_val)
X_test_raw, y_test  = get_X_y(df_test)

scaler = build_scaler(X_train_raw)
X_train = apply_scaler(X_train_raw, scaler)
X_val   = apply_scaler(X_val_raw,   scaler)
X_test  = apply_scaler(X_test_raw,  scaler)

feature_names = list(X_train.columns)
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

## Baseline: Logistic Regression

In [ ]:
from src.models.train import train_baseline
from src.models.evaluate import compute_metrics

lr = train_baseline(X_train, y_train)
lr_proba = lr.predict_proba(X_test)[:, 1]
lr_metrics = compute_metrics(y_test, lr_proba, name='Logistic Regression')
print(lr_metrics)

## XGBoost

In [ ]:
from src.models.train import train_xgboost

xgb = train_xgboost(X_train, y_train, X_val, y_val)
xgb_proba = xgb.predict_proba(X_test)[:, 1]
xgb_metrics = compute_metrics(y_test, xgb_proba, name='XGBoost (raw)')
print(xgb_metrics)

## Calibration

In [ ]:
from src.models.calibrate import calibrate_model

xgb_cal = calibrate_model(xgb, X_val, y_val)
cal_proba = xgb_cal.predict_proba(X_test)[:, 1]
cal_metrics = compute_metrics(y_test, cal_proba, name='XGBoost (calibrated)')
print(cal_metrics)

## Comparison

In [ ]:
from src.models.evaluate import compare_models, plot_roc_curve, plot_calibration_curve
from pathlib import Path

print(compare_models([lr_metrics, xgb_metrics, cal_metrics]))

plot_roc_curve(y_test,
    {'LR': lr_proba, 'XGB': xgb_proba, 'XGB cal': cal_proba},
    save_path=Path('../reports/figures/roc_curves.png'))

plot_calibration_curve(y_test,
    {'XGB raw': xgb_proba, 'XGB calibrated': cal_proba},
    save_path=Path('../reports/figures/calibration_curves.png'))

## Score distribution

In [ ]:
from src.models.calibrate import probability_to_score, score_to_band

scores = probability_to_score(cal_proba)
bands  = [score_to_band(s) for s in scores]

import pandas as pd
result_df = pd.DataFrame({'score': scores, 'band': bands, 'true_label': y_test})
print(result_df['band'].value_counts())

fig, ax = plt.subplots(figsize=(8, 4))
for band, color in [('Low', 'green'), ('Medium', 'orange'), ('High', 'red')]:
    subset = result_df[result_df['band'] == band]['score']
    ax.hist(subset, bins=15, alpha=0.6, label=band, color=color)
ax.set_xlabel('Risk Score')
ax.set_ylabel('Count')
ax.set_title('Risk Score Distribution by Band')
ax.legend()
plt.tight_layout()
plt.show()